In [1]:
import numpy as np
import faiss

In [2]:
class TicketIndexer:
    def __init__(self, vector_dim: int = 384, use_hnsw: bool = True):
        """ Initialises the class whether to use hnsw """
        self.dim = vector_dim
        if use_hnsw:
            # Approximate Nearest Neighbor (HNSW Graph)
            # 32 represents the number of bi-directional links per node
            self.index = faiss.IndexHNSWFlat(vector_dim, 32)
        else:
            # Brute force search
            self.index = faiss.IndexFlatL2(vector_dim)

    def build_index(self, embeddings: np.ndarray):
        """Loads the dense vectors and builds the index."""
        embeddings_flattened = np.ascontiguousarray(embeddings, dtype=np.float32)
        self.index.add(embeddings_flattened)

    def search(self, query_vector: np.ndarray, k: int = 5):
        """Search the query against the built index"""
        query_flattened = np.ascontiguousarray(query_vector, dtype=np.float32)
        distances, indices = self.index.search(query_flattened, k)
        return distances[0], indices[0]
        

In [3]:
X_embed = np.load('../data/processed/embeddings.npy')
X_embed

array([[-0.06532235,  0.08452141,  0.10246462, ..., -0.0042473 ,
        -0.00970798, -0.08990477],
       [-0.05858053,  0.06902478,  0.07796367, ...,  0.00330003,
        -0.0249215 , -0.06884561],
       [-0.05703104,  0.06539226,  0.05661099, ..., -0.0268617 ,
        -0.01657057, -0.05916537],
       ...,
       [ 0.00978868, -0.05098323,  0.04298917, ...,  0.00735625,
        -0.05911245, -0.04495773],
       [ 0.01970189,  0.00922236,  0.05151859, ...,  0.06393553,
        -0.03117783, -0.04156215],
       [-0.02473642,  0.04926911,  0.04493485, ...,  0.02596784,
         0.01846646, -0.0326298 ]], shape=(2000, 384), dtype=float32)

In [25]:
rng = np.random.default_rng()

In [26]:
indices = rng.integers(low=0, high=X_embed.shape[0], size=X_embed.shape[0])

In [28]:
X_train = X_embed[indices[:1000]]
X_test = X_embed[indices[1000:]]
X_train.shape, X_test.shape

((1000, 384), (1000, 384))

In [30]:
indexer = TicketIndexer()
indexer.build_index(X_train)
distances, indices = indexer.search(X_test)

In [34]:
distances, indices

(array([0.        , 0.        , 0.        , 0.39425462, 0.39425462],
       dtype=float32),
 array([104, 827, 996, 464, 938]))